In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
#Check GPU enabled
import torch
import pandas   as pd
import os
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from pathlib import Path
import shutil
print(torch.cuda.is_available())    

True


In [3]:
# Load the sequences and labels for training
loaded_sequences_df = pd.read_pickle('/content/drive/MyDrive/capstone_team_3/data_set/classic_midi_sequences_256.pkl')
print("MIDI sequences loaded from pickle file.")
print(f"Shape of loaded sequences DataFrame: {loaded_sequences_df.shape}")  

MIDI sequences loaded from pickle file.
Shape of loaded sequences DataFrame: (630987, 2)


In [4]:
train_df, test_df = train_test_split(loaded_sequences_df, test_size=0.2, random_state=42)
print(f"Training set size: {len(train_df)}")
print(f"Testing set size: {len(test_df)}")  

Training set size: 504789
Testing set size: 126198


In [5]:
# def cosine_similarity(vec1, vec2):
#     vec1 = np.asarray(vec1, dtype=np.float32)
#     vec2 = np.asarray(vec2, dtype=np.float32)
#     dot_product = np.dot(vec1, vec2)
#     norm_vec1 = np.linalg.norm(vec1)
#     norm_vec2 = np.linalg.norm(vec2)
#     if norm_vec1 == 0 or norm_vec2 == 0:
#         return 0.0
#     return float(dot_product / (norm_vec1 * norm_vec2))

# def features_to_vector(features):
#     features = np.asarray(features, dtype=np.float32)
#     if features.ndim == 1:
#         features = features.reshape(1, -1)
#     summary_vector = np.concatenate([
#         features.mean(axis=0),
#         features.std(axis=0),
#         features.min(axis=0),
#         features.max(axis=0)
#     ])
#     return summary_vector.astype(np.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


### GRU  model 
Prepare normalized sequence tensors, train a simple LSTM classifier, and compare learned embeddings with cosine similarity.

In [6]:
# Prepare normalized tensors and dataloaders
train_sequences = np.stack(train_df['sequence'].to_numpy()).astype(np.float32)
test_sequences = np.stack(test_df['sequence'].to_numpy()).astype(np.float32)

feature_mean = train_sequences.reshape(-1, train_sequences.shape[-1]).mean(axis=0)
feature_std = train_sequences.reshape(-1, train_sequences.shape[-1]).std(axis=0) + 1e-6

# Normalize the sequences
X_train = (train_sequences - feature_mean) / feature_std
X_test = (test_sequences - feature_mean) / feature_std

label_encoder = LabelEncoder()
label_encoder.fit(loaded_sequences_df['label'])
y_train = label_encoder.transform(train_df['label'])
y_test = label_encoder.transform(test_df['label'])

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

batch_size = 64
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=batch_size, shuffle=False)

input_dim = X_train.shape[-1]
sequence_length = X_train.shape[1]
num_classes = len(label_encoder.classes_)

print(f'Train tensor shape: {X_train_tensor.shape}')
print(f'Test tensor shape: {X_test_tensor.shape}')
print(f'Input dimension: {input_dim}, Sequence length: {sequence_length}, Classes: {num_classes}')

Train tensor shape: torch.Size([504789, 256, 3])
Test tensor shape: torch.Size([126198, 256, 3])
Input dimension: 3, Sequence length: 256, Classes: 289


In [7]:
 #Define and train a baseline LSTM classifier
class GRUSequenceClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, num_layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        _, hidden_state = self.gru(x)
        embedding = self.dropout(hidden_state[-1])
        logits = self.classifier(embedding)
        return logits


In [8]:
# Hyperparameters
gru_hidden_dim = 128
gru_learning_rate = 1e-3
gru_epochs = 5


In [9]:
# Initialize model, loss function, and optimizer
gru_model = GRUSequenceClassifier(
    input_dim=input_dim,
    hidden_dim=gru_hidden_dim,
    num_classes=num_classes
).to(device)

gru_criterion = nn.CrossEntropyLoss()
gru_optimizer = optim.Adam(gru_model.parameters(), lr=gru_learning_rate)


In [10]:
gru_history = []
for epoch in range(gru_epochs):
    gru_model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch_inputs, batch_labels in train_loader:
        batch_inputs = batch_inputs.to(device)
        batch_labels = batch_labels.to(device)

        gru_optimizer.zero_grad()
        logits = gru_model(batch_inputs)
        loss = gru_criterion(logits, batch_labels)
        loss.backward()
        gru_optimizer.step()

        train_loss += loss.item() * batch_inputs.size(0)
        train_correct += (logits.argmax(dim=1) == batch_labels).sum().item()
        train_total += batch_labels.size(0)

    gru_model.eval()
    test_loss = 0.0
    test_correct = 0
    test_total = 0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for batch_inputs, batch_labels in test_loader:
            batch_inputs = batch_inputs.to(device)
            batch_labels = batch_labels.to(device)

            logits = gru_model(batch_inputs)
            loss = gru_criterion(logits, batch_labels)

            preds = logits.argmax(dim=1)
            test_loss += loss.item() * batch_inputs.size(0)
            test_correct += (preds == batch_labels).sum().item()
            test_total += batch_labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(batch_labels.cpu().numpy())

    epoch_metrics = {
        'epoch': epoch + 1,
        'train_loss': train_loss / train_total,
        'train_accuracy': train_correct / train_total,
        'test_loss': test_loss / test_total,
        'test_accuracy': test_correct / test_total,
        'test_accuracy_sklearn': accuracy_score(all_targets, all_preds)
    }
    gru_history.append(epoch_metrics)
    print(
        f"GRU Epoch {epoch + 1}/{gru_epochs} - "
        f"train_loss: {epoch_metrics['train_loss']:.4f}, "
        f"train_accuracy: {epoch_metrics['train_accuracy']:.4f}, "
        f"test_loss: {epoch_metrics['test_loss']:.4f}, "
        f"test_accuracy: {epoch_metrics['test_accuracy']:.4f}"
    )

gru_history_df = pd.DataFrame(gru_history)
print('\nGRU training complete.')
gru_history_df

GRU Epoch 1/5 - train_loss: 2.1942, train_accuracy: 0.4723, test_loss: 0.4956, test_accuracy: 0.8798
GRU Epoch 2/5 - train_loss: 0.3542, train_accuracy: 0.9041, test_loss: 0.0558, test_accuracy: 0.9896
GRU Epoch 3/5 - train_loss: 0.1255, train_accuracy: 0.9657, test_loss: 0.0764, test_accuracy: 0.9826
GRU Epoch 4/5 - train_loss: 0.0686, train_accuracy: 0.9808, test_loss: 0.0228, test_accuracy: 0.9941
GRU Epoch 5/5 - train_loss: 0.0489, train_accuracy: 0.9862, test_loss: 0.0064, test_accuracy: 0.9984

GRU training complete.


,epoch,train_loss,train_accuracy,test_loss,test_accuracy,test_accuracy_sklearn
0,1,2.194216,0.472330,0.495645,0.879808,0.879808
1,2,0.354232,0.904128,0.055812,0.989635,0.989635
2,3,0.125527,0.965701,0.076450,0.982623,0.982623
3,4,0.068633,0.980816,0.022817,0.994073,0.994073
4,5,0.048852,0.986222,0.006428,0.998399,0.998399


In [11]:
# Evaluate GRU predictions on the test set and compare with LSTM results
gru_model.eval()
gru_predictions = []
with torch.no_grad():
    for batch_inputs, batch_labels in test_loader:
        batch_inputs = batch_inputs.to(device)
        batch_labels = batch_labels.to(device)
        logits = gru_model(batch_inputs)
        predictions = logits.argmax(dim=1)
        gru_predictions.extend(predictions.cpu().numpy())   

In [12]:
# Test set accuracy for GRU
gru_test_accuracy = accuracy_score(all_targets, gru_predictions)
print(f'GRU Test Accuracy: {gru_test_accuracy:.4f}')

GRU Test Accuracy: 0.9984


In [ ]:
# PR

Test accuracy: 0.9979


,true_label,predicted_label
0,burg_spinnerlied.mid,burg_spinnerlied.mid
1,bor_ps7.mid,bor_ps7.mid
2,liz_et_trans4.mid,liz_et_trans4.mid
3,appass_1.mid,appass_1.mid
4,liz_et_trans4.mid,liz_et_trans4.mid
5,liz_et6.mid,liz_et6.mid
6,mz_570_1.mid,mz_570_1.mid
7,schub_d960_1.mid,schub_d960_1.mid
8,brahms_opus1_4.mid,brahms_opus1_4.mid
9,schumm-2.mid,schumm-2.mid


In [15]:
sample_index = 0
sample_sequence = X_test_tensor[sample_index:sample_index + 1].to(device)
sample_true_label = label_encoder.inverse_transform([y_test[sample_index]])[0]
import sklearn.metrics.pairwise  as pairwise
with torch.no_grad():
    sample_logits, sample_embedding = model(sample_sequence, return_embedding=True)
    sample_prediction = int(sample_logits.argmax(dim=1).item())
    sample_embedding = sample_embedding.squeeze(0).cpu().numpy()
# Sample embedding 
print(f'Sample true label: {sample_true_label}')
print(f"Sample predicted label: {label_encoder.inverse_transform([sample_prediction])[0]}")
print( "sample embedding shape:", sample_embedding.shape)
similarity_rows = []
for class_index, class_name in enumerate(label_encoder.classes_):
    similarity_rows.append({
        'label': class_name,
        'cosine_similarity': pairwise.cosine_similarity(sample_embedding.reshape(1, -1), prototype_embeddings[class_index].reshape(1, -1))[0][0]
    })



Sample true label: burg_spinnerlied.mid
Sample predicted label: burg_spinnerlied.mid
sample embedding shape: (128,)


In [16]:
similarity_df = pd.DataFrame(similarity_rows).sort_values('cosine_similarity', ascending=False).head(5)
print(f'Sample true label: {sample_true_label}')
print(f"Sample predicted label: {label_encoder.inverse_transform([sample_prediction])[0]}")
similarity_df

Sample true label: burg_spinnerlied.mid
Sample predicted label: burg_spinnerlied.mid


,label,cosine_similarity
58,burg_spinnerlied.mid,0.783696
89,chpn_op23.mid,0.545800
243,schuim-2.mid,0.525158
270,scn16_7.mid,0.513604
205,mz_331_1.mid,0.475677


In [14]:
# Load the sequences and labels for testing on external MIDI files
external_test_df = pd.read_pickle('/content/drive/MyDrive/capstone_team_3/data_set/external_test_sequences.pkl')
print("External MIDI sequences loaded from pickle file.")
print(f"Shape of external test sequences DataFrame: {external_test_df.shape}")
print("Sample external test sequences:")
external_test_df.head()

External MIDI sequences loaded from pickle file.
Shape of external test sequences DataFrame: (203723, 2)
Sample external test sequences:


,sequence,label
0,"[[0.7999999999999998, 0.0, 44.0], [0.300000000...",Piano Sonata n12 K332.mid
1,"[[0.30000000000000027, 4.0, 47.0], [0.79999999...",Piano Sonata n12 K332.mid
2,"[[0.7999999999999998, 3.0, 49.0], [0.304166666...",Piano Sonata n12 K332.mid
3,"[[0.3041666666666667, -3.0, 52.0], [0.79999999...",Piano Sonata n12 K332.mid
4,"[[0.7999999999999998, 1.0, 54.0], [0.299999999...",Piano Sonata n12 K332.mid


In [15]:
X_test_external= np.stack(external_test_df['sequence'].to_numpy()).astype(np.float32)
X_test_external = (X_test_external - feature_mean) / feature_std

In [16]:
# Test with the merged test sequences and predicted labels from the GRU model
gru_test_predictions = []
with torch.no_grad():
    for seq in X_test_external:
        seq_tensor = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)
        logits = gru_model(seq_tensor)
        predicted_label_index = int(logits.argmax(dim=1).item())
        gru_test_predictions.append(label_encoder.inverse_transform([predicted_label_index])[0])

In [17]:
# Add predictions to  Xtest_df
test_df['gru_predicted_label'] = gru_test_predictions
print("Sample test sequences with GRU predicted labels:")
print(test_df.head())

ValueError: Length of values (203723) does not match length of index (126198)

In [ ]:
# Stoe the  results in a pickle file for later analysis (only labels and predicted labels)
external_test_df[['label', 'predicted_label']].to_pickle('/content/drive/MyDrive/capstone_team_3/data_set/external_test_predictions.pkl')
print("External test predictions saved to pickle file.")

In [ ]:
# Store the GRU model state for later use
gru_model_save_path = '/content/drive/MyDrive/capstone_team_3/models/gru_sequence_classifier.pth'
torch.save(gru_model.state_dict(), gru_model_save_path)
print(f"GRU model state saved to {gru_model_save_path}")